<a href="https://colab.research.google.com/github/foxtrot-0715/my-first-data-project/blob/main/notebooks/hw4_eda_and_validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Ячейка 1: Импорт необходимых библиотек под требования Data-Centric MLOps
import pandas as pd
import numpy as np
from pydantic import BaseModel, Field, ValidationError
from sklearn.model_selection import StratifiedKFold
from typing import Optional

print("Шаг 1 выполнен: Библиотеки успешно загружены. Пайплайн готов.")

Шаг 1 выполнен: Библиотеки успешно загружены. Пайплайн готов.


In [ ]:
# Ячейка 2: Создание схемы жесткой валидации данных согласно чек-листу лекции
class NomenclatureRecord(BaseModel):
    id: int = Field(..., gt=0, description="ID записи в 1С должен быть строго положительным")
    raw_text: str = Field(..., min_length=3, max_length=1000, description="Длина наименования должна быть адекватной")
    category_id: int = Field(..., ge=1, le=79, description="Всего доступно строго 79 номенклатурных категорий")
    manufacturer_part_number: Optional[str] = Field(None, max_length=100)

print("Шаг 2 выполнен: Pydantic-валидатор данных успешно инициализирован.")

Шаг 2 выполнен: Pydantic-валидатор данных успешно инициализирован.


In [ ]:
# Ячейка 3: Генерация и симуляция «грязного» датасета номенклатуры (Имитация EDA)
import numpy as np

# Фиксируем сид для воспроизводимости экспериментов (требование слайда 7)
np.random.seed(42)
n_samples = 1000

# Симулируем жесткий дисбаланс классов (Long-tail распределение из лекции)
style_classes = list(range(1, 80))
# Задаем вероятности: топ-5 категорий забирают 30% всего объема, остальные 74 делят остаток
p_distribution = [0.3 / 5 if i <= 5 else 0.7 / 74 for i in style_classes]

categories = np.random.choice(style_classes, size=n_samples, p=p_distribution)

# Базовый шаблон промышленных наименований
raw_texts = [
    f"Кабель силовой медный ВВГнг-LS {np.random.randint(100, 999)}х{np.random.randint(1, 10)}"
    for _ in range(n_samples)
]

df = pd.DataFrame({
    "id": list(range(1, n_samples + 1)),
    "raw_text": raw_texts,
    "category_id": categories
})

# Внедряем аномалии и технический мусор для проверки нашего MLOps-валидатора
df.loc[10, "raw_text"] = "А"  # Критически короткая запись (ошибка закупщика)
df.loc[20, "category_id"] = 99  # Несуществующая категория (шум и ошибка разметки аналитика)

print(f"Шаг 3 выполнен. Тестовый датасет успешно сгенерирован. Формат: {df.shape}")
print(f"\nВывод EDA — Уровень дисбаланса (Топ-5 категорий-гигантов):")
print(df['category_id'].value_counts().head(5))

Шаг 3 выполнен. Тестовый датасет успешно сгенерирован. Формат: (1000, 3)

Вывод EDA — Уровень дисбаланса (Топ-5 категорий-гигантов):
category_id
2    73
3    68
5    64
1    63
4    51
Name: count, dtype: int64


In [ ]:
# Ячейка 4: Запуск пайплайна валидации данных через Pydantic
valid_records = []
corrupted_records = []

# Итерируемся по строкам нашего датасета
for idx, row in df.iterrows():
    try:
        # Пытаемся упаковать каждую строчку в жесткую Pydantic-схему
        record = NomenclatureRecord(
            id=int(row["id"]),
            raw_text=str(row["raw_text"]),
            category_id=int(row["category_id"])
        )
        valid_records.append(record.dict())
    except ValidationError as e:
        # Если данные нарушают правила — перехватываем ошибку и отправляем в брак
        corrupted_records.append({
            "id": row["id"],
            "raw_text": row["raw_text"],
            "category_id": row["category_id"],
            "error_details": e.errors()[0]['msg']  # Вытаскиваем понятную причину ошибки
        })

# Собираем чистый датасет для обучения модели
df_clean = pd.DataFrame(valid_records)
df_corrupted = pd.DataFrame(corrupted_records)

print("Шаг 4 выполнен: Валидация данных завершена успешно!")
print(f"--> Чистых записей допущено к обучению модели: {len(df_clean)}")
print(f"--> Изолировано грязных записей для ручного аудита экспертами НСИ: {len(df_corrupted)}")

# Выводим на экран то, что заблокировал наш валидатор
print("\nЛог заблокированных записей (Брак):")
print(df_corrupted[["id", "category_id", "error_details"]])

Шаг 4 выполнен: Валидация данных завершена успешно!
--> Чистых записей допущено к обучению модели: 998
--> Изолировано грязных записей для ручного аудита экспертами НСИ: 2

Лог заблокированных записей (Брак):
   id  category_id                             error_details
0  11            1  String should have at least 3 characters
1  21           99  Input should be less than or equal to 79


/tmp/ipykernel_9611/3571404179.py:14: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  valid_records.append(record.dict())


In [ ]:
# Ячейка 5: Стратифицированное разбиение для надежной валидации при дисбалансе
from sklearn.model_selection import StratifiedKFold

# Настраиваем кросс-валидацию на 5 фолдов со случайным перемешиванием
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Выделяем признаки и целевую метку из нашего чистого датасета
X = df_clean["raw_text"].values
y = df_clean["category_id"].values

print("Шаг 5 выполнен: Стратегия валидации успешно запущена.\n")
print(f"Общий объем чистых данных для кросс-валидации: {len(X)} записей.")

# Демонстрируем распределение по фолдам
for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"--> Фолд {fold+1}: Размер обучающей выборки (Train) = {len(train_idx)}, Размер проверочной (Val) = {len(val_idx)}")

Шаг 5 выполнен: Стратегия валидации успешно запущена.

Общий объем чистых данных для кросс-валидации: 998 записей.
--> Фолд 1: Размер обучающей выборки (Train) = 798, Размер проверочной (Val) = 200
--> Фолд 2: Размер обучающей выборки (Train) = 798, Размер проверочной (Val) = 200
--> Фолд 3: Размер обучающей выборки (Train) = 798, Размер проверочной (Val) = 200
--> Фолд 4: Размер обучающей выборки (Train) = 799, Размер проверочной (Val) = 199
--> Фолд 5: Размер обучающей выборки (Train) = 799, Размер проверочной (Val) = 199


/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 3 members, which is less than n_splits=5.
  warnings.warn(
